# 🌦️ Project 09 – Weather Classifier

**Goal:** Predict the weather type (Sunny / Rainy / Cloudy / Snowy) from meteorological measurements using a **Random Forest** classifier.

| | |
|---|---|
| **Difficulty** | ⭐⭐ Beginner–Intermediate |
| **Estimated time** | 45 – 60 minutes |
| **Key concepts** | Random Forest, Feature Engineering, Confusion Matrix, Cross-Validation |

---

## What you will learn
1. How to generate realistic synthetic data for a classification task
2. How to engineer new features from raw measurements
3. How to train and evaluate a Random Forest classifier
4. How to interpret feature importances and a confusion matrix

## Step 1 — Imports

We use **scikit-learn** for the model, **pandas / numpy** for data wrangling, and **matplotlib / seaborn** for charts.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_recall_fscore_support
)

print('Imports OK ✓')

## Step 2 — Generate Weather Data

We create a **synthetic** dataset of 1 000 daily weather observations.  
Each row has five raw features:

| Feature | Range | Unit |
|---|---|---|
| temperature | –10 → 40 | °C |
| humidity | 10 → 100 | % |
| wind_speed | 0 → 80 | km/h |
| cloud_cover | 0 → 100 | % |
| pressure | 970 → 1040 | hPa |

The **label** (Sunny / Rainy / Cloudy / Snowy) is assigned using simple meteorological rules — this gives the model a real pattern to learn.

In [ ]:
rng = np.random.default_rng(42)
N = 1000

temperature  = rng.uniform(-10, 40,   N)
humidity     = rng.uniform(10,  100,  N)
wind_speed   = rng.uniform(0,   80,   N)
cloud_cover  = rng.uniform(0,   100,  N)
pressure     = rng.uniform(970, 1040, N)

# Feature engineering: derived meteorological indices
heat_index = temperature + 0.33 * (humidity / 100 * 6.105) - 4
discomfort = heat_index - 0.55 * (1 - humidity / 100) * (heat_index - 14.5)

# Rule-based labels
labels = []
for temp, hum, wind, cloud, pres in zip(temperature, humidity, wind_speed, cloud_cover, pressure):
    if temp <= 2 and hum > 60 and cloud > 70:
        labels.append('Snowy')
    elif hum > 70 and cloud > 60 and pres < 1005:
        labels.append('Rainy')
    elif cloud > 50 and hum > 50:
        labels.append('Cloudy')
    else:
        labels.append('Sunny')

df = pd.DataFrame({
    'temperature_c':   temperature,
    'humidity_pct':    humidity,
    'wind_speed_kmh':  wind_speed,
    'cloud_cover_pct': cloud_cover,
    'pressure_hpa':    pressure,
    'heat_index':      heat_index,
    'discomfort':      discomfort,
    'weather_type':    labels,
})

print(df.shape)
df.head()

### Class balance

Let's check how many samples we have per weather type.

In [ ]:
print(df['weather_type'].value_counts())
df['weather_type'].value_counts().plot(kind='bar', color=['gold','steelblue','grey','skyblue'], edgecolor='black', rot=0)
plt.title('Weather Type Distribution')
plt.xlabel('Weather Type')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## Step 3 — Prepare Data for Modelling

scikit-learn needs numbers, so we **label-encode** the target column (e.g. `Cloudy → 0`, `Rainy → 1`, …), then split into train / test sets.

In [ ]:
feature_cols = [c for c in df.columns if c != 'weather_type']

X = df[feature_cols].values
le = LabelEncoder()
y = le.fit_transform(df['weather_type'])

print('Classes:', le.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## Step 4 — Train Random Forest

A **Random Forest** builds many decision trees on random subsets of the data and features, then votes on the final prediction. This makes it robust and hard to overfit.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,   # 100 trees
    max_depth=8,        # each tree goes at most 8 levels deep
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.2%}')

## Step 5 — Evaluate the Model

### 5a. Classification report
Shows **precision**, **recall**, and **F1-score** for each class.

In [ ]:
print(classification_report(y_test, y_pred, target_names=le.classes_))

### 5b. Confusion matrix

Each cell (row i, col j) shows how many times a sample of class *i* was predicted as class *j*. Perfect predictions sit on the diagonal.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

### 5c. Cross-validation (5-fold)

Cross-validation gives a more reliable accuracy estimate by testing on 5 different splits.

In [ ]:
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
print(f'5-Fold CV: {cv_scores.mean():.2%} ± {cv_scores.std():.2%}')
print(f'Per-fold : {[f"{s:.2%}" for s in cv_scores]}')

## Step 6 — Feature Importance

Random Forest can tell us which features matter most for predictions. This is incredibly useful in real projects.

In [ ]:
importance_df = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)
importance_df.plot(kind='barh', figsize=(8, 5), color='steelblue', edgecolor='white')
plt.title('Feature Importances')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print(importance_df.sort_values(ascending=False))

## Step 7 — Make a Prediction

Now let's predict the weather for a single new day.

In [ ]:
# New day: 28°C, 45% humidity, 15 km/h wind, 20% cloud, 1018 hPa
temp, hum, wind, cloud, pres = 28, 45, 15, 20, 1018
hi = temp + 0.33 * (hum / 100 * 6.105) - 4
dis = hi - 0.55 * (1 - hum / 100) * (hi - 14.5)

sample = np.array([[temp, hum, wind, cloud, pres, hi, dis]])
pred = le.inverse_transform(rf.predict(sample))[0]
proba = rf.predict_proba(sample)[0]

print(f'Prediction  : {pred}')
print(f'Probabilities:')
for cls, p in zip(le.classes_, proba):
    bar = '█' * int(p * 30)
    print(f'  {cls:8s}  {bar} {p:.1%}')

---

## 🛠️ Try It Yourself

1. **Change the rules** in Step 2 and retrain — does accuracy change?
2. **Try fewer trees** (`n_estimators=10`) and compare accuracy vs speed.
3. **Add noise** to the features with `rng.normal(0, 5, N)` and see how robust the model is.
4. **Replace Random Forest** with `sklearn.tree.DecisionTreeClassifier` — what changes?
5. **Print the most important tree** using `sklearn.tree.export_text(rf.estimators_[0], feature_names=feature_cols)`.

---

## 📝 Concepts Summary

| Concept | Plain English |
|---|---|
| **Random Forest** | A crowd of decision trees that vote together — more reliable than a single tree |
| **Feature Engineering** | Creating new, more informative columns from raw data (e.g. heat index from temperature + humidity) |
| **Train/Test Split** | Keep some data hidden during training to test how well the model generalises |
| **Confusion Matrix** | A grid showing correct vs. incorrect predictions per class |
| **Cross-Validation** | Averaging accuracy over multiple train/test splits for a fairer estimate |
| **Feature Importance** | How much each feature contributed to the model's decisions |
| **Precision** | Of all the times we predicted 'Rainy', how often were we right? |
| **Recall** | Of all the truly 'Rainy' days, how many did we catch? |